# Semantic FAQ Search

## Introduction

The objective of this mini-project is to explore semantic search using sentence embeddings and cosine similarity. I match a user's question to the most relevant question in an FAQ dataset and return its stored answer.

This experiment uses **240 synthetic FAQ entries** about programming, data science, and machine learning. It is a retrieval system, not a generative chatbot.

## Loading the Data

The `faq.csv` file contains an ID, category, question, and answer for each entry. I use IDs to connect retrieval results with their original answers. I also prepared `evaluation_queries.csv` with 36 separately phrased questions and expected FAQ IDs for a small retrieval evaluation.

In [ ]:
# Install the libraries required for semantic retrieval
%pip install sentence-transformers scikit-learn

In [ ]:
# Import the required libraries
import csv
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


def data_file(name):
    """Find a data file when running from the notebook folder or repository root."""
    for path in (Path(name), Path("nlp/semantic-faq-search") / name):
        if path.is_file():
            return path
    raise FileNotFoundError(f"Could not find {name}. Run the notebook from its folder or the repository root.")

In [ ]:
# Load the FAQ records
with data_file("faq.csv").open(encoding="utf-8", newline="") as file:
    faq_rows = list(csv.DictReader(file))

assert faq_rows and all(all(row.get(key) for key in ("id", "category", "question", "answer")) for row in faq_rows)
assert len({row["id"] for row in faq_rows}) == len(faq_rows)

faq_questions = [row["question"] for row in faq_rows]
print(f"Loaded {len(faq_rows)} FAQ entries across {len(set(row['category'] for row in faq_rows))} categories.")

## Modeling

I use the pretrained `sentence-transformers/all-MiniLM-L6-v2` model to convert questions into embeddings. I compare a user query with all FAQ questions using cosine similarity, rank the most similar entries, and return the stored answer. The threshold is an adjustable heuristic, not a calibrated confidence score.

In [ ]:
# Load the sentence embedding model and encode the FAQ questions
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
faq_embeddings = model.encode(faq_questions, convert_to_numpy=True, show_progress_bar=True)

In [ ]:
# Retrieve the most similar questions and their existing answers
def search_faq(user_question, top_k=3, threshold=0.45):
    """Return up to top_k matches above a cosine-similarity threshold."""
    query_embedding = model.encode([user_question], convert_to_numpy=True)
    scores = cosine_similarity(query_embedding, faq_embeddings)[0]
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    matches = []
    for index in ranked_indices:
        if scores[index] < threshold:
            continue
        row = faq_rows[index]
        matches.append({
            "id": row["id"],
            "category": row["category"],
            "question": row["question"],
            "answer": row["answer"],
            "similarity": float(scores[index]),
        })
    return matches


def faq_response(user_question):
    matches = search_faq(user_question, top_k=1)
    if not matches:
        return "Sorry, I could not find a relevant answer to that question."
    return matches[0]["answer"]

## Retrieval Evaluation

I created a separate set of 36 paraphrased questions with known relevant FAQ IDs. I use these to check whether the intended answer appears first or within the top three search results. The evaluation computes Top-1 accuracy, Recall@3, and mean reciprocal rank (MRR).

This small, synthetic evaluation set is useful for learning, but it is not a measure of production performance or accuracy on real user queries.

In [ ]:
# Evaluate retrieval without a similarity threshold
with data_file("evaluation_queries.csv").open(encoding="utf-8", newline="") as file:
    evaluation_rows = list(csv.DictReader(file))

assert evaluation_rows and all(row["query"] and row["relevant_id"] for row in evaluation_rows)
faq_ids = [row["id"] for row in faq_rows]
assert all(row["relevant_id"] in faq_ids for row in evaluation_rows)

queries = [row["query"] for row in evaluation_rows]
query_embeddings = model.encode(queries, convert_to_numpy=True)
query_scores = cosine_similarity(query_embeddings, faq_embeddings)

first_hits = 0
hits_at_3 = 0
reciprocal_ranks = []
for case, scores in zip(evaluation_rows, query_scores):
    order = np.argsort(scores)[::-1]
    retrieved_ids = [faq_ids[i] for i in order]
    rank = retrieved_ids.index(case["relevant_id"]) + 1
    first_hits += rank == 1
    hits_at_3 += rank <= 3
    reciprocal_ranks.append(1 / rank)

n = len(evaluation_rows)
print(f"Evaluation queries: {n}")
print(f"Top-1 accuracy: {first_hits / n:.1%}")
print(f"Recall@3: {hits_at_3 / n:.1%}")
print(f"Mean reciprocal rank: {np.mean(reciprocal_ranks):.3f}")

## Testing

I try several differently worded questions and inspect the highest-ranked FAQ entry and its similarity score. This helps identify cases where related terminology is mistaken for the intended meaning.

In [ ]:
# Inspect example retrieval results
example_questions = [
    "I installed something with pip but Python cannot import it. Why?",
    "How can I assess whether an image detector is good?",
    "Can I search for answers using different wording?",
]
for question in example_questions:
    matches = search_faq(question, top_k=1, threshold=0.0)
    match = matches[0]
    print(f"Query: {question}")
    print(f"Matched question: {match['question']}")
    print(f"Similarity: {match['similarity']:.3f}")
    print(f"Answer: {match['answer']}\n")

## Interactive Search

The following optional cell allows me to type questions and retrieve stored FAQ answers. Type `exit` to stop.

In [ ]:
# Optional command-line interaction
print("FAQ Search")
print("Type 'exit' to quit.\n")

while True:
    question = input("Question: ").strip()
    if question.lower() == "exit":
        print("Goodbye.")
        break
    if not question:
        continue
    print("Answer:", faq_response(question))

## Conclusion

This mini-project helped me understand how a sentence embedding model can retrieve related questions even when the wording differs. I also added a small labeled evaluation set to compare retrieval quality using ranking metrics.

The FAQ entries and test queries are synthetic. Similarity thresholds need further calibration, and the results should not be treated as evidence of performance on real support conversations.